# Bedrock Module · Day 13 — Bedrock Agents: Build & Test

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

So far your model can *talk* (Day 11) and *look things up* (Day 12 RAG). A **Bedrock Agent** lets it **act**: you give it **action groups** — tools backed by AWS Lambda functions — described by an **OpenAPI schema**, and the agent decides *which* tool to call, with *what* arguments, to satisfy a request. It runs a **reason → act → observe** loop, keeps **session memory** across turns, and emits a **trace** of its chain-of-thought you can inspect. Today you build one by hand so every part is visible.

> **Keyless & offline.** No AWS account, no Lambda. We register plain Python functions as the 'action group', describe them with an OpenAPI-shaped schema, and drive them through a **`FakeBedrockAgent`** with the same `invoke_agent` shape (a completion + a trace + a session id). Swap in `boto3.client("bedrock-agent-runtime")` and the concepts map one-to-one.


## 0. Setup — config from Day-13 Python

The agent's region/model come from a settings object (this morning's lesson), not literals.


In [1]:
import re, json

class AgentSettings:                     # stand-in for a pydantic BaseSettings
    aws_region  = "eu-west-2"
    agent_model = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"
    big_amount  = 10_000                 # policy threshold used by a tool

cfg = AgentSettings()
print("agent config:", cfg.aws_region, "|", cfg.agent_model.split('.')[-1])


agent config: eu-west-2 | claude-sonnet-4-5-20250929-v1:0


## 1. Action group — the tools the agent can call

An **action group** is a set of functions the agent may invoke. In production each is an AWS Lambda; here each is a Python function. They're just normal code — the agent never sees the body, only the **schema** (next cell) that tells it what each does and what arguments it takes.


In [2]:
# a tiny 'account database' so the tools have something to act on
ACCOUNTS = {
    "ACC-1001": {"name": "A. Khan",  "balance": 4200.0,  "status": "active"},
    "ACC-2002": {"name": "B. Okoro", "balance": 55000.0, "status": "active"},
}

def get_balance(account_id: str):
    acc = ACCOUNTS.get(account_id)
    if not acc: return {"error": f"no such account {account_id}"}
    return {"account_id": account_id, "balance": acc["balance"], "currency": "GBP"}

def transfer_funds(account_id: str, amount: float):
    acc = ACCOUNTS.get(account_id)
    if not acc: return {"error": f"no such account {account_id}"}
    if amount > acc["balance"]: return {"error": "insufficient funds"}
    # policy: large transfers need human approval (tie-in to HITL, Day 05)
    if amount >= cfg.big_amount:
        return {"status": "NEEDS_APPROVAL", "amount": amount, "reason": "over 10k threshold"}
    acc["balance"] -= amount
    return {"status": "OK", "new_balance": acc["balance"]}

ACTIONS = {"get_balance": get_balance, "transfer_funds": transfer_funds}
print("action group:", list(ACTIONS))


action group: ['get_balance', 'transfer_funds']


☝ Two tools: a safe read (`get_balance`) and a state-changing action (`transfer_funds`) that already encodes a **policy** — transfers ≥ £10k return `NEEDS_APPROVAL` instead of executing. Keeping policy *inside* the tool means the agent can't talk its way past it.


## 2. OpenAPI schema — how the agent **understands** the tools

The agent can't read your Python; it reads a **schema**. Bedrock Agents use an **OpenAPI 3.0** spec per action group: each operation has a description, parameters, and types. Good descriptions = the agent picks the right tool — this is prompt engineering for tools.


In [3]:
OPENAPI = {
  "openapi": "3.0.0",
  "info": {"title": "AccountActions", "version": "1.0.0"},
  "paths": {
    "/get_balance": {"post": {
       "description": "Get the current balance for a customer account.",
       "parameters": [{"name": "account_id", "required": True, "schema": {"type": "string"}}]}},
    "/transfer_funds": {"post": {
       "description": "Move money out of an account. Amounts over 10000 GBP require approval.",
       "parameters": [
         {"name": "account_id", "required": True, "schema": {"type": "string"}},
         {"name": "amount",     "required": True, "schema": {"type": "number"}}]}},
  },
}
print("operations described:", list(OPENAPI["paths"]))


operations described: ['/get_balance', '/transfer_funds']


☝ Each `description` is what the model reads to decide *when* to use a tool and *how* to fill its parameters. Note the transfer description **states the £10k rule** — telling the agent the policy up front makes it ask for approval proactively, not just react to the tool's response.


## 3. The agent loop — reason → act → observe

A Bedrock Agent runs an internal loop: read the user's goal, **reason** about which action to take, **call** it, **observe** the result, and either loop again or answer. Our `FakeBedrockAgent` does a simple keyword-based reasoner (a real one is the LLM) but produces the **same `invoke_agent` output shape**: a final `completion` plus a `trace` of every step.


In [4]:
class FakeBedrockAgent:
    def __init__(self, actions, schema, settings):
        self.actions=actions; self.schema=schema; self.cfg=settings
        self.memory={}                                  # sessionId -> list of turns

    def _reason(self, text, session):
        """Pick an action + args from the request (a real agent uses the LLM here)."""
        acc = (re.search(r'ACC-\d+', text) or [None])[0] if re.search(r'ACC-\d+', text) else None
        # carry the account from earlier turns in this session (memory!)
        acc = acc or self.memory.get(session, {}).get("last_account")
        amt = re.search(r'(\d[\d,]*)\s*(?:gbp|pounds)?', text.lower())
        amount = float(amt.group(1).replace(',', '')) if ('transfer' in text.lower() and amt) else None
        # only call a tool once we actually know the account (else ask for it)
        if 'balance' in text.lower()  and acc:            return ("get_balance", {"account_id": acc})
        if 'transfer' in text.lower() and acc and amount: return ("transfer_funds", {"account_id": acc, "amount": amount})
        return (None, {})

    def invoke_agent(self, inputText, sessionId):
        trace=[]
        trace.append({"step":"rationale","text":f"User said: {inputText!r}. Deciding an action."})
        action, args = self._reason(inputText, sessionId)
        if action is None:
            trace.append({"step":"rationale","text":"No tool needed; answering directly."})
            return {"completion":"I can check a balance or make a transfer. Give me an account id.",
                    "trace":trace, "sessionId":sessionId}
        trace.append({"step":"invocationInput","actionGroup":action,"parameters":args})
        result = self.actions[action](**args)
        trace.append({"step":"observation","actionGroup":action,"result":result})
        # remember the account for follow-up turns
        if args.get("account_id"):
            self.memory.setdefault(sessionId, {})["last_account"] = args["account_id"]
        completion = self._verbalise(action, result)
        trace.append({"step":"rationale","text":f"Observed {result}; composing reply."})
        return {"completion":completion, "trace":trace, "sessionId":sessionId}

    def _verbalise(self, action, result):
        if result.get("error"):                  return f"Sorry — {result['error']}."
        if action=="get_balance":                return f"The balance is {result['balance']:.2f} {result['currency']}."
        if result.get("status")=="NEEDS_APPROVAL": return f"That transfer of {result['amount']:.0f} needs manager approval (over 10k)."
        if action=="transfer_funds":             return f"Done — new balance {result['new_balance']:.2f} GBP."
        return str(result)

agent = FakeBedrockAgent(ACTIONS, OPENAPI, cfg)
print("agent ready")


agent ready


☝ Note the structure of `invoke_agent`'s output: a human `completion` **and** a structured `trace` (rationale → invocationInput → observation). That trace is the agent's chain-of-thought — the thing you read to debug *why* it did what it did.


## 4. Turn 1 — a read action, with its trace

Ask for a balance. Watch the agent pick `get_balance`, call it, observe, and answer — and print the **reasoning trace** alongside the reply.


In [5]:
def show(resp):
    print("AGENT:", resp["completion"])
    print("trace:")
    for t in resp["trace"]:
        if t["step"]=="invocationInput": print(f"   ⚙ call {t['actionGroup']}({t['parameters']})")
        elif t["step"]=="observation":   print(f"   👁 result {t['result']}")
        else:                            print(f"   💭 {t['text']}")
    print()

SID = "session-alpha"
show(agent.invoke_agent("What's the balance on ACC-2002?", sessionId=SID))


AGENT: The balance is 55000.00 GBP.
trace:
   💭 User said: "What's the balance on ACC-2002?". Deciding an action.
   ⚙ call get_balance({'account_id': 'ACC-2002'})
   👁 result {'account_id': 'ACC-2002', 'balance': 55000.0, 'currency': 'GBP'}
   💭 Observed {'account_id': 'ACC-2002', 'balance': 55000.0, 'currency': 'GBP'}; composing reply.



☝ Four trace steps show the whole decision: it reasoned, called `get_balance(account_id='ACC-2002')`, observed the dict, and verbalised it. In the real console this is the **Trace** tab — your primary tool for understanding and debugging an agent.


## 5. Turn 2 — **session memory** (no account repeated)

Same `sessionId`, and this time the user **doesn't repeat the account number** — they just say 'transfer 500'. The agent pulls `ACC-2002` from **session memory** set on turn 1. Memory is what makes a multi-turn conversation feel coherent.


In [6]:
show(agent.invoke_agent("Now transfer 500 from it", sessionId=SID))


AGENT: Done — new balance 54500.00 GBP.
trace:
   💭 User said: 'Now transfer 500 from it'. Deciding an action.
   ⚙ call transfer_funds({'account_id': 'ACC-2002', 'amount': 500.0})
   👁 result {'status': 'OK', 'new_balance': 54500.0}
   💭 Observed {'status': 'OK', 'new_balance': 54500.0}; composing reply.



☝ The agent resolved 'it' to `ACC-2002` because the session remembered the last account — the transfer went through and the balance dropped. Bedrock Agents enable this with **memory** on the session; drop the `sessionId` and each turn would start cold.


## 6. Turn 3 — policy in action: a **big** transfer pauses

Now ask for a £20,000 transfer. The `transfer_funds` tool's own policy returns `NEEDS_APPROVAL`, and the agent surfaces that instead of moving money — a guardrail the model can't override because it lives in the tool, not the prompt.


In [7]:
show(agent.invoke_agent("transfer 20,000 from ACC-2002", sessionId=SID))

# a fresh session has no memory of the account -> agent asks for it
show(agent.invoke_agent("what's my balance?", sessionId="session-beta"))


AGENT: That transfer of 20000 needs manager approval (over 10k).
trace:
   💭 User said: 'transfer 20,000 from ACC-2002'. Deciding an action.
   ⚙ call transfer_funds({'account_id': 'ACC-2002', 'amount': 20000.0})
   👁 result {'status': 'NEEDS_APPROVAL', 'amount': 20000.0, 'reason': 'over 10k threshold'}
   💭 Observed {'status': 'NEEDS_APPROVAL', 'amount': 20000.0, 'reason': 'over 10k threshold'}; composing reply.

AGENT: I can check a balance or make a transfer. Give me an account id.
trace:
   💭 User said: "what's my balance?". Deciding an action.
   💭 No tool needed; answering directly.



☝ Two safety behaviours: the £20k transfer stopped at `NEEDS_APPROVAL` (this is exactly where you'd wire the Day-05 human-in-the-loop gate), and a **new session** with no memory correctly asked for an account instead of guessing. Isolation between sessions is a security property, not an accident.


## 7. How this maps to a real Bedrock Agent

| You built | Bedrock Agent equivalent |
|---|---|
| `ACTIONS` Python fns | **Action group** backed by **AWS Lambda** |
| `OPENAPI` dict | the action group's **OpenAPI 3.0 schema** |
| `_reason()` | the **foundation model** (Claude) doing the reasoning |
| `invoke_agent()` | `bedrock-agent-runtime.invoke_agent(...)` |
| `trace` list | the **Trace** (rationale / invocationInput / observation) |
| `self.memory[sessionId]` | agent **session memory** |
| `NEEDS_APPROVAL` | a **return-of-control** / confirmation step |

```python
import boto3
rt = boto3.client("bedrock-agent-runtime", region_name="eu-west-2")
resp = rt.invoke_agent(agentId="AG123", agentAliasId="LIVE",
                       sessionId="session-alpha", inputText="balance on ACC-2002?",
                       enableTrace=True)
# stream resp['completion'] chunks; read resp trace events for the reasoning
```


## 8. Recap — you built an acting agent

| Piece | Role |
|---|---|
| **action group** | the tools the agent may call |
| **OpenAPI schema** | how the model understands each tool |
| **reason→act→observe** | the agent's decision loop |
| **trace** | inspectable chain-of-thought (debug here) |
| **session memory** | continuity across turns |
| **tool-side policy** | guardrails the model can't bypass |

**Barclays lens:** a Bedrock Agent turns an LLM into something that *does* work against real systems — and the trace + tool-side policies + session isolation are what make that safe to put near money.

